In [1]:
import pandas as pd

df = pd.read_csv("BankChurners.csv")
df.head()

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,769911858,Existing Customer,40,F,4,High School,NaN,Less than $40K,Blue,34,...,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000


In [2]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns)

Index(['clientnum', 'attrition_flag', 'customer_age', 'gender',
       'dependent_count', 'education_level', 'marital_status',
       'income_category', 'card_category', 'months_on_book',
       'total_relationship_count', 'months_inactive_12_mon',
       'contacts_count_12_mon', 'credit_limit', 'total_revolving_bal',
       'avg_open_to_buy', 'total_amt_chng_q4_q1', 'total_trans_amt',
       'total_trans_ct', 'total_ct_chng_q4_q1', 'avg_utilization_ratio'],
      dtype='object')


## Convert churn into risk levels

In [3]:
def classify_risk(flag):
    if flag == "Attrited Customer":
        return "High"
    else:
        return "Low"

df["risk_level"] = df["attrition_flag"].apply(classify_risk)

df.head()

,clientnum,attrition_flag,customer_age,gender,dependent_count,education_level,marital_status,income_category,card_category,months_on_book,...,contacts_count_12_mon,credit_limit,total_revolving_bal,avg_open_to_buy,total_amt_chng_q4_q1,total_trans_amt,total_trans_ct,total_ct_chng_q4_q1,avg_utilization_ratio,risk_level
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,Low
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,Low
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,Low
3,769911858,Existing Customer,40,F,4,High School,NaN,Less than $40K,Blue,34,...,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,Low
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,Low


## Count Risk Levels

In [4]:
risk_counts = df["risk_level"].value_counts()
print(risk_counts)

risk_level
Low     8500
High    1627
Name: count, dtype: int64


## High-risk customers

In [5]:
high_risk = df[df["risk_level"] == "High"]
high_risk.head()

,clientnum,attrition_flag,customer_age,gender,dependent_count,education_level,marital_status,income_category,card_category,months_on_book,...,contacts_count_12_mon,credit_limit,total_revolving_bal,avg_open_to_buy,total_amt_chng_q4_q1,total_trans_amt,total_trans_ct,total_ct_chng_q4_q1,avg_utilization_ratio,risk_level
21,708508758,Attrited Customer,62,F,0,Graduate,Married,Less than $40K,Blue,49,...,3,1438.3,0,1438.3,1.047,692,16,0.600,0.000,High
39,708300483,Attrited Customer,66,F,0,Doctorate,Married,abc,Blue,56,...,3,7882.0,605,7277.0,1.052,704,16,0.143,0.077,High
51,779471883,Attrited Customer,54,F,1,Graduate,Married,Less than $40K,Blue,40,...,1,1438.3,808,630.3,0.997,705,19,0.900,0.562,High
54,714374133,Attrited Customer,56,M,2,Graduate,Married,$120K +,Blue,36,...,3,15769.0,0,15769.0,1.041,602,15,0.364,0.000,High
61,712030833,Attrited Customer,48,M,2,Graduate,Married,$60K - $80K,Silver,35,...,4,34516.0,0,34516.0,0.763,691,15,0.500,0.000,High


## 1. Risk Segmentation

In [6]:
# create a simple churn score using available behavior columns
df["churn_score"] = (
    df["months_inactive_12_mon"] * 0.30 +
    df["contacts_count_12_mon"] * 0.20 +
    df["total_revolving_bal"] / df["credit_limit"] * 0.30 +
    df["avg_utilization_ratio"] * 0.20
)

df[["months_inactive_12_mon", "contacts_count_12_mon", "avg_utilization_ratio", "churn_score"]].head()

,months_inactive_12_mon,contacts_count_12_mon,avg_utilization_ratio,churn_score
0,1,3,0.061,0.930567
1,1,2,0.105,0.752395
2,1,0,0.000,0.300000
3,4,1,0.760,1.779920
4,1,0,0.000,0.300000


## 2. Risk Level Classification

In [7]:
def classify_risk(score):
    if score >= 2.5:
        return "High"
    elif score >= 1.5:
        return "Medium"
    else:
        return "Low"

df["risk_level"] = df["churn_score"].apply(classify_risk)

df[["churn_score", "risk_level"]].head(10)

,churn_score,risk_level
0,0.930567,Low
1,0.752395,Low
2,0.300000,Low
3,1.779920,Medium
4,0.300000,Low
5,0.855492,Low
6,0.932878,Low
7,1.024001,Low
8,0.656382,Low
9,1.571962,Medium


## Count customers by risk level

In [8]:
risk_counts = df["risk_level"].value_counts()
print(risk_counts)

risk_level
Low       6578
Medium    3483
High        66
Name: count, dtype: int64


## 3. High-Risk Customer List

In [9]:
high_risk = df[df["risk_level"] == "High"].copy()

high_risk.head()

,clientnum,attrition_flag,customer_age,gender,dependent_count,education_level,marital_status,income_category,card_category,months_on_book,...,credit_limit,total_revolving_bal,avg_open_to_buy,total_amt_chng_q4_q1,total_trans_amt,total_trans_ct,total_ct_chng_q4_q1,avg_utilization_ratio,risk_level,churn_score
313,783542133,Existing Customer,53,F,3,Graduate,Married,Less than $40K,Blue,47,...,2759.0,2223,536.0,1.320,2093,42,1.000,0.806,High,2.602918
373,714363858,Existing Customer,54,F,4,High School,Married,Less than $40K,Blue,36,...,1594.0,1411,183.0,0.817,1459,35,1.188,0.885,High,2.542558
477,789337233,Existing Customer,63,M,1,Graduate,Married,$40K - $60K,Blue,53,...,2880.0,2362,518.0,0.890,1323,26,0.444,0.820,High,2.610042
1184,783020958,Existing Customer,39,M,3,NaN,Married,$80K - $120K,Blue,33,...,18582.0,794,17788.0,0.717,1836,46,0.586,0.043,High,2.621419
1327,778734108,Existing Customer,36,M,3,Uneducated,Married,$60K - $80K,Blue,16,...,1686.0,1252,434.0,0.575,1758,42,1.100,0.743,High,2.671376


In [10]:
cols_to_show = [
    "clientnum",
    "attrition_flag",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "avg_utilization_ratio",
    "churn_score",
    "risk_level"
]

high_risk[cols_to_show].head(10)

,clientnum,attrition_flag,months_inactive_12_mon,contacts_count_12_mon,avg_utilization_ratio,churn_score,risk_level
313,783542133,Existing Customer,6,2,0.806,2.602918,High
373,714363858,Existing Customer,5,3,0.885,2.542558,High
477,789337233,Existing Customer,6,2,0.820,2.610042,High
1184,783020958,Existing Customer,6,4,0.043,2.621419,High
1327,778734108,Existing Customer,5,4,0.743,2.671376,High
1336,769579608,Existing Customer,6,4,0.715,2.957420,High
1386,771602883,Existing Customer,5,4,0.415,2.507355,High
1450,756409758,Existing Customer,5,4,0.648,2.624043,High
1479,778882758,Existing Customer,5,4,0.559,2.579457,High
1569,719049033,Existing Customer,6,4,0.118,2.658920,High


## 4. Summary Statistics

In [11]:
summary = f"""
Churn Risk Summary
------------------
High Risk Customers: {risk_counts.get("High", 0)}
Medium Risk Customers: {risk_counts.get("Medium", 0)}
Low Risk Customers: {risk_counts.get("Low", 0)}
Total Customers: {len(df)}
"""

print(summary)


Churn Risk Summary
------------------
High Risk Customers: 66
Medium Risk Customers: 3483
Low Risk Customers: 6578
Total Customers: 10127



## Add extra business stats

In [12]:
high_pct = round((risk_counts.get("High", 0) / len(df)) * 100, 2)
medium_pct = round((risk_counts.get("Medium", 0) / len(df)) * 100, 2)
low_pct = round((risk_counts.get("Low", 0) / len(df)) * 100, 2)

print(f"High Risk %: {high_pct}%")
print(f"Medium Risk %: {medium_pct}%")
print(f"Low Risk %: {low_pct}%")

High Risk %: 0.65%
Medium Risk %: 34.39%
Low Risk %: 64.96%


## 5. Save Results to CSV

In [13]:
df.to_csv("churn_scored_customers.csv", index=False)
high_risk.to_csv("high_risk_customers.csv", index=False)

print("Files saved:")
print("- churn_scored_customers.csv")
print("- high_risk_customers.csv")

Files saved:
- churn_scored_customers.csv
- high_risk_customers.csv


In [14]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\tahmi
['.anaconda', '.cache', '.conda', '.condarc', '.continuum', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.matplotlib', '.ms-ad', '.spss', '.virtual_documents', '.vscode', '.wdm', '1', '2', 'anaconda3', 'AppData', 'Application Data', 'BankChurners.csv', 'CHI_data_collection.ipynb', 'CHI_project', 'CHI_SEO_analysis-Copy1.ipynb', 'CHI_SEO_analysis_clean.ipynb', 'churn_risk_analysis.ipynb', 'churn_scored_customers.csv', 'Contacts', 'Cookies', 'Datensicherung.bat', 'Desktop', 'Docum', 'Downloads', 'Favorites', 'fitness_CHI_v2', 'high_risk_customers.csv', 'HTML Part 1', 'Links', 'llm_prompt.txt', 'Local Settings', 'Local Sites', 'Microsoft', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{f87c9ee0-e8eb-11ef-96ac-c6cb65e9ff24}.TM.blf', 'NTUSER.DAT{f87c9ee0-e8eb-11ef-96ac-c6cb65e9ff24}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{f87c9ee0-e8eb-11ef-96ac-c6cb65e9ff24}.TMContainer00000000000000000002.

## 6. AI Insights

In [15]:
llm_prompt = f"""
You are a business analyst.

Based on this churn risk summary, provide:
1. A short executive summary
2. Three business insights
3. Three recommended actions

Data:
- Total customers: {len(df)}
- High risk customers: {risk_counts.get("High", 0)} ({high_pct}%)
- Medium risk customers: {risk_counts.get("Medium", 0)} ({medium_pct}%)
- Low risk customers: {risk_counts.get("Low", 0)} ({low_pct}%)

Also consider that the dataset includes inactivity, contact frequency, and utilization behavior.
"""

print(llm_prompt)


You are a business analyst.

Based on this churn risk summary, provide:
1. A short executive summary
2. Three business insights
3. Three recommended actions

Data:
- Total customers: 10127
- High risk customers: 66 (0.65%)
- Medium risk customers: 3483 (34.39%)
- Low risk customers: 6578 (64.96%)

Also consider that the dataset includes inactivity, contact frequency, and utilization behavior.



## Save AI prompt to text file

In [16]:
with open("llm_prompt.txt", "w", encoding="utf-8") as f:
    f.write(llm_prompt)

print("Saved: llm_prompt.txt")

Saved: llm_prompt.txt


In [17]:
df.to_csv("churn_scored_customers.csv", index=False)

In [18]:
ai_output = """
Executive Summary:
Customer churn risk is relatively low overall, with the majority of customers in the low-risk segment...

Business Insights:
1. Customer inactivity strongly correlates with churn risk.
2. Medium-risk customers represent the largest opportunity segment.
3. Utilization behavior indicates disengagement patterns.

Recommended Actions:
1. Launch re-engagement campaigns for medium-risk customers.
2. Provide targeted incentives for high-risk customers.
3. Improve customer interaction frequency.
"""

In [19]:
with open("ai_insights.txt", "w", encoding="utf-8") as f:
    f.write(ai_output)

print("Saved: ai_insights.txt")

Saved: ai_insights.txt
